In [2]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
from astroquery.gaia import Gaia
from scipy.optimize import minimize
from isochrones.mist import MIST_Isochrone
from scipy.interpolate import RegularGridInterpolator
mist = MIST_Isochrone()

Some IP addresses of users launching heavy query showers have temporarily been disabled. Please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk) for advice. Workaround solutions for the Gaia Archive issues following the infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


PyMultiNest not imported.  MultiNest fits will not work.


In [3]:
# helper function that converts ra and dec to distance (pc)

def get_distance_pc(ra, dec, radius=2/3600):
    """
    Query Gaia for parallax near RA/Dec.
    Returns distance in parsecs.
    """
    
    query = f"""
    SELECT TOP 1 parallax
    FROM gaiadr3.gaia_source
    WHERE CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {ra}, {dec}, {radius})
    ) = 1
    """
    
    job = Gaia.launch_job(query)
    result = job.get_results()
    
    if len(result) == 0 or result['parallax'][0] <= 0:
        return np.nan
    
    parallax_mas = result['parallax'][0]
    distance_pc = 1000.0 / parallax_mas
    
    return distance_pc

In [4]:
df1 = pd.read_csv("/Users/elenadickens/Desktop/ASTR_502/ASTR502_Master_Target_List.csv")
df2 = pd.read_csv("/Users/elenadickens/Desktop/ASTR_502/ASTR502_Master_Photometry_List.csv")

df1 = df1.drop_duplicates(subset="hostname")
df2 = df2.drop_duplicates(subset="hostname")

merged = pd.merge(df1, df2, on="hostname", how="inner")

merged["ra"] = merged["ra_x"]
merged["dec"] = merged["dec_x"]

merged = merged.drop(columns=["ra_x", "ra_y", "dec_x", "dec_y", "pl_name_y"])

merged = merged[[
    "hostname",
    "st_teff", "st_age", "st_met",
    "ra", "dec",
    "gaia_Gmag", "gaia_BPmag", "gaia_RPmag",
    "Jmag", "Hmag", "Kmag",
    "e_Jmag", "e_Hmag", "e_Kmag"
]]

merged = merged.dropna(subset=[
    "st_teff", "st_age", "st_met",
    "ra", "dec",
    "gaia_Gmag", "gaia_BPmag", "gaia_RPmag",
    "Jmag", "Hmag", "Kmag",
    "e_Jmag", "e_Hmag", "e_Kmag"
])

merged = merged[merged["st_age"] < 1.0]

merged["st_met"] = merged["st_met"].fillna(0.0)
merged["e_Jmag"] = merged["e_Jmag"].fillna(0.03)
merged["e_Hmag"] = merged["e_Hmag"].fillna(0.03)
merged["e_Kmag"] = merged["e_Kmag"].fillna(0.03)

# merged_sample = merged.sample(n=500, random_state=42)

merged["distance_pc"] = merged.apply(
    lambda row: get_distance_pc(row["ra"], row["dec"]),
    axis=1
)

display(HTML(merged.to_html(max_rows=50)))

,hostname,st_teff,st_age,st_met,ra,dec,gaia_Gmag,gaia_BPmag,gaia_RPmag,Jmag,Hmag,Kmag,e_Jmag,e_Hmag,e_Kmag,distance_pc
11,TOI-6016,6110.0,0.3000,0.2800,4.523960,59.766198,11.909994,12.331106,11.320911,10.629,10.308,10.224,0.021,0.015,0.016,363.747188
14,Qatar-4,5215.0,0.1700,0.1030,4.859275,44.027597,13.400405,13.867678,12.771313,12.030,11.631,11.519,0.020,0.021,0.020,327.154515
22,Qatar-5,5747.0,0.5300,0.3770,7.053937,42.061345,12.511989,12.887348,11.975026,11.348,11.089,10.956,0.023,0.021,0.020,372.259051
29,WASP-93,6700.0,0.7000,0.0700,9.458744,51.288778,10.984386,11.209536,10.591513,10.233,9.997,9.942,0.022,0.021,0.016,372.001405
66,TOI-2046,6250.0,0.4500,-0.0600,16.184942,74.331305,11.408128,11.718330,10.933183,10.415,10.125,10.094,0.024,0.031,0.023,287.947853
110,TOI-2152,6630.0,0.8300,0.2820,26.338972,77.790123,11.243453,11.661012,10.635723,9.973,9.669,9.597,0.026,0.030,0.024,320.863187
120,TOI-262,5310.0,0.8000,0.2600,32.534687,-31.070627,8.677938,9.109069,8.081018,7.387,7.051,6.967,0.023,0.040,0.020,44.143328
136,GPX-1,7000.0,0.2700,0.3500,38.369165,56.025708,12.212760,12.461465,11.815224,11.357,11.221,11.181,0.026,0.024,0.021,676.566346
155,HD 18599,5145.0,0.4000,0.0000,44.262014,-56.191869,8.741666,9.190199,8.127134,7.428,7.029,6.883,0.018,0.015,0.020,38.632868
171,WASP-139,5310.0,0.5000,0.2000,49.562173,-41.302023,12.264109,12.701511,11.666922,10.982,10.575,10.472,0.023,0.023,0.021,212.83671


In [5]:
age_grid  = np.linspace(6.0, 10.2, 60)     # log(age/yr) from 1 Myr → 16 Gyr
feh_grid  = np.linspace(-2.5, 0.5, 31)     # halo → super-solar
mass_grid = np.linspace(0.1, 10.0, 200)    # M dwarfs → massive MS

interpolator_bands = [
    'J_mag','H_mag','K_mag',
    'G_mag','BP_mag','RP_mag',
    'Teff'
]
shape = (len(age_grid), len(feh_grid), len(mass_grid))
cubes = {band: np.full(shape, np.nan) for band in interpolator_bands}

for i, age in enumerate(age_grid):
    for j, feh in enumerate(feh_grid):
        iso = mist.isochrone(age, feh)
        if iso.empty:
            continue
        iso = iso.sort_values('initial_mass')
        for band in interpolator_bands:
            cubes[band][i, j, :] = np.interp(
                mass_grid,
                iso['initial_mass'],
                iso[band],
                left=np.nan,
                right=np.nan
            )

interpolators = {}
for band in interpolator_bands:
    interpolators[band] = RegularGridInterpolator(
        (age_grid, feh_grid, mass_grid),
        cubes[band],
        bounds_error=False,
        fill_value=np.nan
    )

In [6]:
# bands we will use
bands = ["G_mag", "BP_mag", "RP_mag", "J_mag", "H_mag", "K_mag"]

# Define which bands to fit and mapping to interpolator bands
fit_bands = ["G", "BP", "RP", "J", "H", "K"]
band_map = {b: b + "_mag" for b in fit_bands}  # maps G -> G_mag etc.

def chi2_model(params, star):

    logage, mass = params
    feh = np.clip(star['feh'], feh_grid.min(), feh_grid.max())

    # observed photometry
    y = np.array([star['photometry'][b] for b in fit_bands])
    yerr = np.array([star['phot_err'].get(b,0.02) for b in fit_bands])

    # model magnitudes
    model_mags = np.array([
        interpolators[band_map[b]]([[logage, feh, mass]])[0]
        for b in fit_bands
    ])

    if np.any(np.isnan(model_mags)):
        return 1e10

    chi2_phot = np.sum(((y - model_mags)/yerr)**2)

    # ---- Teff prior ----
    Teff_obs = star["Teff"]
    Teff_err = star.get("Teff_err", 100)   # default uncertainty if not provided

    Teff_model = interpolators["Teff"]([[logage, feh, mass]])[0]

    if np.isnan(Teff_model):
        return 1e10

    chi2_teff = ((Teff_obs - Teff_model)/Teff_err)**2

    return chi2_phot + chi2_teff

In [7]:
def fit_star(star):

    initial_guess = [7.5,1.0]
    bounds = [
        (6.0,10.2),
        (0.1,10.0)
    ]
    result = minimize(
        chi2_model,
        initial_guess,
        args=(star,),
        bounds=bounds,
        method='L-BFGS-B'
    )
    best_logage, best_mass = result.x

    return {
    'hostname': star['hostname'],

    # fitted parameters
    'age_fit': 10**best_logage / 1e9,   # Gyr
    'mass_fit': best_mass,

    # catalog parameters
    'age_catalog': star['age_true'],
    'feh': star['feh'],

    # observed magnitudes
    'G_abs': star['photometry']['G'],
    'BP_abs': star['photometry']['BP'],
    'RP_abs': star['photometry']['RP'],
    'J_abs': star['photometry']['J'],
    'H_abs': star['photometry']['H'],
    'K_abs': star['photometry']['K']
    }

In [8]:
def row_to_star(row):
    star = {
        "hostname": row["hostname"],
        "photometry": {
            "G": row["gaia_Gmag"],
            "BP": row["gaia_BPmag"],
            "RP": row["gaia_RPmag"],
            "J": row["Jmag"],
            "H": row["Hmag"],
            "K": row["Kmag"],
        },
        "phot_err": {
            "J": row["e_Jmag"],
            "H": row["e_Hmag"],
            "K": row["e_Kmag"],
        },
        "Teff": row["st_teff"],
        "feh": row["st_met"],
        "age_true": row.get("st_age", None)  # for validation only
    }
    return star

star_dicts = [row_to_star(row) for _, row in merged.iterrows()]
results = []
for star in star_dicts[]:
    try:
        fit = fit_star(star)
        results.append(fit)
    except Exception as e:
        print("Failed:", star['hostname'], e)

results_df = pd.DataFrame(results)
results_df

,hostname,age_fit,mass_fit,age_catalog,feh,G_abs,BP_abs,RP_abs,J_abs,H_abs,K_abs
0,TOI-6016,0.022884,0.150001,0.30,0.280,11.909994,12.331106,11.320911,10.629,10.308,10.224
1,Qatar-4,1.979254,0.150003,0.17,0.103,13.400405,13.867678,12.771313,12.030,11.631,11.519
2,Qatar-5,0.014385,0.149800,0.53,0.377,12.511989,12.887348,11.975026,11.348,11.089,10.956
3,WASP-93,1.355824,0.191199,0.70,0.070,10.984386,11.209536,10.591513,10.233,9.997,9.942
4,TOI-2046,2.216985,0.173247,0.45,-0.060,11.408128,11.718330,10.933183,10.415,10.125,10.094
...,...,...,...,...,...,...,...,...,...,...,...
95,Kepler-236,0.428614,0.149800,0.91,-0.050,15.357136,16.314177,14.385992,13.224,12.537,12.312
96,Kepler-1760,2.208739,0.162749,0.60,0.080,14.350877,14.718285,13.689777,13.040,12.628,12.537
97,Kepler-1747,0.018331,0.150278,0.70,0.210,15.958084,16.409569,15.345313,14.654,14.230,14.036
98,KOI-351,2.004700,0.149845,0.53,0.098,13.727177,14.025368,13.273602,12.790,12.531,12.482
